# Exibindo as tabelas dentro do volume

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/projeto/data_schema/data_volume"))

path,name,size,modificationTime
dbfs:/Volumes/projeto/data_schema/data_volume/Chamados_Hora.csv,Chamados_Hora.csv,2628992,1763606412000
dbfs:/Volumes/projeto/data_schema/data_volume/base_atendentes.csv,base_atendentes.csv,422,1763606411000
dbfs:/Volumes/projeto/data_schema/data_volume/base_motivos.csv,base_motivos.csv,570,1763606411000
dbfs:/Volumes/projeto/data_schema/data_volume/canais.csv,canais.csv,136,1763606411000
dbfs:/Volumes/projeto/data_schema/data_volume/chamados.csv,chamados.csv,2675735,1763606412000
dbfs:/Volumes/projeto/data_schema/data_volume/clientes.csv,clientes.csv,1264875,1763606412000
dbfs:/Volumes/projeto/data_schema/data_volume/custos.csv,custos.csv,733814,1763606412000
dbfs:/Volumes/projeto/data_schema/data_volume/pesquisa_satisfacao.csv,pesquisa_satisfacao.csv,395128,1763606412000


# Importando as bibliotecas

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

# Criando a tabela bronze

In [0]:
%sql
-- DROP DATABASE bronze CASCADE; -- Executar se tiver uma bronze já criada
CREATE DATABASE IF NOT EXISTS bronze;

# Lendo os arquivos csv para a camada bronze

In [0]:
# Base path inside your Unity Catalog volume
base_path = "dbfs:/Volumes/projeto/data_schema/data_volume/"

# Mapping of file names to Bronze table names
files_to_tables = {
    "Chamados_Hora.csv": "bronze.chamados_hora",
    "base_atendentes.csv": "bronze.base_atendentes",
    "base_motivos.csv": "bronze.base_motivos",
    "canais.csv": "bronze.canais",
    "chamados.csv": "bronze.chamados",
    "clientes.csv": "bronze.clientes",
    "custos.csv": "bronze.custos",
    "pesquisa_satisfacao.csv": "bronze.pesquisa_satisfacao"
}


for file_name, table_name in files_to_tables.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{base_path}{file_name}")
        .withColumn("ingestion_timestamp", current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(table_name)
    print(f" Table created: {table_name}")   

 Table created: bronze.chamados_hora
 Table created: bronze.base_atendentes
 Table created: bronze.base_motivos
 Table created: bronze.canais
 Table created: bronze.chamados
 Table created: bronze.clientes
 Table created: bronze.custos
 Table created: bronze.pesquisa_satisfacao


# Exibindo as tabelas criadas

In [0]:
%sql
SHOW TABLES IN bronze;

database,tableName,isTemporary
bronze,base_atendentes,false
bronze,base_motivos,false
bronze,canais,false
bronze,chamados,false
bronze,chamados_hora,false
bronze,clientes,false
bronze,custos,false
bronze,pesquisa_satisfacao,false
